# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review the available record sets and fields by their `@id`.
We will print out the `@id` and labels for all discovered record sets and their fields.

In [ ]:
print("Available Record Sets:")
record_set_ids = list(dataset.record_sets.keys())
for rec_id in record_set_ids:
    rec = dataset.record_sets[rec_id]
    print(f"- @id: {rec_id}, name: {getattr(rec, 'name', None)}")
    print("  Fields:")
    for fid in rec.fields:
        field = dataset.fields[fid]
        print(f"    - @id: {fid}, name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Let's load data from each record set using their `@id`. We will store each table in a `dataframes` dictionary by record set `@id`. You can swap out which record set to use for further exploration.

In [ ]:
# Extract data from each record set
record_sets = record_set_ids  # all discovered record set @ids
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame with {len(dataframes[record_set_id])} rows and columns: {dataframes[record_set_id].columns.tolist()}")
    else:
        print(f"No records found for record set {record_set_id}.")

# As an example, pick the first record set for demonstration
if record_set_ids:
    example_record_set = record_set_ids[0]
    if example_record_set in dataframes:
        print(f"\nPreview of record set: {example_record_set}")
        display(dataframes[example_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic processing such as filtering records, normalization, and basic grouping. For demonstration, we'll identify a numeric field in the example record set.

In [ ]:
import numpy as np

# Identify a numeric field in the chosen record set
example_df = dataframes[example_record_set]
numeric_fields = example_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns: {numeric_fields}")

# Select the first numeric field for illustration
if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    raise ValueError(f"No numeric fields found in record set {example_record_set}")

threshold = example_df[numeric_field].mean()
filtered_df = example_df[example_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records (first 5 rows):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt grouping by first non-numeric, non-unique field if available
non_numeric_fields = [col for col in example_df.columns if col not in numeric_fields and example_df[col].nunique() < len(example_df)]
if non_numeric_fields:
    group_field = non_numeric_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped {numeric_field} mean by {group_field} (first 5 groups):")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its normalized version, along with a group bar chart if grouping was possible.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Histogram of the numeric field
plt.figure(figsize=(9, 4))
plt.subplot(1,2,1)
example_df[numeric_field].hist(bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")

plt.subplot(1,2,2)
filtered_df[f"{numeric_field}_normalized"].hist(bins=30)
plt.title(f"Normalized {numeric_field} (filtered)")
plt.xlabel(f"{numeric_field}_normalized")
plt.tight_layout()
plt.show()

# Grouped bar plot if applicable
if 'grouped_df' in locals():
    plt.figure(figsize=(8,4))
    plt.bar(grouped_df[group_field].astype(str), grouped_df[numeric_field])
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We successfully loaded and explored the FAIR^2 dataset using its Croissant schema.

- **Loaded dataset and explored its metadata and available record sets by `@id`.**
- **Extracted records into pandas DataFrames and previewed the data.**
- **Performed basic EDA including filtering, normalization, and grouping by fields.**
- **Visualized key numeric field distributions and group means.**

For further analysis, you can access more record sets or fields by swapping the `@id`s in the relevant cells above.